In [1]:
import os
import re
import time
import shutil
import statistics
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


# LangChain imports — all correct for v0.2+
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    CSVLoader,
    UnstructuredExcelLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [2]:
LOADER_MAP = {
    '.pdf':  PyPDFLoader,
    '.txt':  TextLoader,
    '.docx': Docx2txtLoader,
    '.csv':  CSVLoader,
    '.xlsx': UnstructuredExcelLoader,
    '.xls':  UnstructuredExcelLoader,
}

In [3]:
# Dataset and configuration paths
DATASET_DIR = Path('dataset')

In [4]:
def load_documents(directory: Path):
    """Load all supported documents from the dataset directory."""
    all_docs = []
    for filepath in directory.iterdir():
        ext = filepath.suffix.lower()
        if ext not in LOADER_MAP:
            print(f'⚠️  Skipping: {filepath.name}')
            continue
        try:
            loader = LOADER_MAP[ext](str(filepath))
            docs   = loader.load()
            for doc in docs:
                doc.metadata['source']    = filepath.name
                doc.metadata['file_type'] = ext
                doc.metadata['loaded_at'] = datetime.now().isoformat()
            all_docs.extend(docs)
            print(f'Loaded: {filepath.name} → {len(docs)} page(s)')
        except Exception as e:
            print(f'Error loading {filepath.name}: {e}')
    print(f'\n Total documents loaded: {len(all_docs)}')
    return all_docs
raw_documents = load_documents(DATASET_DIR)

Loaded: ai_overview.txt → 1 page(s)

 Total documents loaded: 1


In [5]:
if raw_documents:
    print('\n--- Preview ---')
    print(raw_documents[0].page_content[:300])
    print('Metadata:', raw_documents[0].metadata)


--- Preview ---
Artificial Intelligence and Machine Learning Overview

Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, and self-correction.

Machine Learning (ML) is a subset of AI that allows syste
Metadata: {'source': 'ai_overview.txt', 'file_type': '.txt', 'loaded_at': '2026-03-29T20:42:36.414860'}


Text Preprocessing

In [6]:
def clean_text(text: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

Chunking

In [7]:
def chunk_documents(documents, chunk_size=800, chunk_overlap=150):
    """Split documents into overlapping chunks for better retrieval."""
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' ', ''],
        length_function=len,
    )
    chunks = splitter.split_documents(documents)
    for i, chunk in enumerate(chunks):
        chunk.metadata['chunk_id'] = i
    return chunks

In [8]:
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 150

chunks = chunk_documents(raw_documents, CHUNK_SIZE, CHUNK_OVERLAP)

In [9]:
print(f' Total chunks : {len(chunks)}')
print(f'   Chunk size   : {CHUNK_SIZE}')
print(f'   Overlap      : {CHUNK_OVERLAP}')

 Total chunks : 3
   Chunk size   : 800
   Overlap      : 150


In [10]:
print('\n--- Sample Chunk ---')
print(chunks[0].page_content)

lengths = [len(c.page_content) for c in chunks]
print(f'\n Min:{min(lengths)}  Max:{max(lengths)}  Mean:{statistics.mean(lengths):.0f}  Median:{statistics.median(lengths):.0f}')


--- Sample Chunk ---
Artificial Intelligence and Machine Learning Overview

Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, and self-correction.

Machine Learning (ML) is a subset of AI that allows systems to learn and improve from experience without being explicitly programmed. ML algorithms build mathematical models based on sample data, known as "training data", in order to make predictions or decisions.

Types of Machine Learning:
1. Supervised Learning: Uses labeled training data to learn input-output patterns
2. Unsupervised Learning: Finds hidden patterns in unlabeled data
3. Reinforcement Learning: Learns through trial and error with rewards and penalties

 Min:505  Max:768  Mean:665  Median:722


Embedding Generation

In [11]:
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

In [12]:
print(f'Loading: {EMBEDDING_MODEL} ...')
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)
print('Embedding model loaded')

Loading: sentence-transformers/all-MiniLM-L6-v2 ...


C:\Users\techr\AppData\Local\Temp\ipykernel_7916\100806032.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6237.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [13]:
test_vec = embedding_model.embed_query('What is RAG?')
print(f'Embedding dimensions : {len(test_vec)}')
print(f'Sample values        : {[round(v,4) for v in test_vec[:5]]}')

Embedding dimensions : 384
Sample values        : [-0.0696, 0.0952, 0.016, 0.0068, -0.0884]


FAISS Vector Store

In [14]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
print(f'FAISS index built')

FAISS index built


In [15]:
print(f'   Vectors stored : {vectorstore.index.ntotal}')
print(f'   Dimensions     : {vectorstore.index.d}')

   Vectors stored : 3
   Dimensions     : 384


Save and reload FAISS index

In [16]:
FAISS_INDEX_PATH = 'faiss_index'
vectorstore.save_local(FAISS_INDEX_PATH)
print(f'Saved to: {FAISS_INDEX_PATH}/')

vectorstore = FAISS.load_local(
    FAISS_INDEX_PATH,
    embedding_model,
    allow_dangerous_deserialization=True
)
print(f'Reloaded: {vectorstore.index.ntotal} vectors')

Saved to: faiss_index/
Reloaded: 3 vectors


Query Processing & Similarity Search

In [17]:
TOP_K = 4

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': TOP_K},
)

In [18]:
def retrieve_relevant_chunks(query: str, k: int = TOP_K):
    """Embed query and retrieve top-K most similar chunks with scores."""
    return vectorstore.similarity_search_with_score(query, k=k)


test_query = 'What is RAG and how does it work?'
results    = retrieve_relevant_chunks(test_query)

print(f'Query: "{test_query}"\n')
for i, (doc, score) in enumerate(results, 1):
    print(f'--- Chunk {i} | Score: {score:.4f} | Source: {doc.metadata.get("source")} ---')
    print(doc.page_content[:200])
    print()

Query: "What is RAG and how does it work?"

--- Chunk 1 | Score: 1.5594 | Source: ai_overview.txt ---
Deep Learning is a subset of ML using neural networks with many layers. It has revolutionized fields like computer vision, natural language processing, and speech recognition.

Natural Language Proces

--- Chunk 2 | Score: 1.6912 | Source: ai_overview.txt ---
FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors, commonly used in RAG systems for fast document retrieval.

Sentence Transformers are

--- Chunk 3 | Score: 1.8829 | Source: ai_overview.txt ---
Artificial Intelligence and Machine Learning Overview

Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. These processes include 



Prompt Engineering

In [19]:
def build_prompt(query: str, context_docs: list, chat_history: list, max_history_turns: int = 3) -> str:
    """Build a structured RAG prompt: system + context + history + query."""
    system = (
        'You are a precise, helpful AI Knowledge Assistant. '
        'Answer questions ONLY using the provided context. '
        'If the answer is not in the context, say: '
        '"I could not find relevant information in the uploaded documents." '
        'Do not fabricate information.'
    )

    context_parts = []
    for i, doc in enumerate(context_docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        context_parts.append(f'[Source {i}: {source}]\n{doc.page_content}')
    context_str = '\n\n---\n\n'.join(context_parts)

    recent = chat_history[-(max_history_turns * 2):]
    history_str = '\n'.join(
        f"{'User' if m['role']=='user' else 'Assistant'}: {m['content']}"
        for m in recent
    ) or 'None'

    return f"""{system}

=== RETRIEVED CONTEXT ===
{context_str}

=== CONVERSATION HISTORY ===
{history_str}

=== CURRENT QUESTION ===
{query}

=== ANSWER ===
Provide a clear, factual answer based strictly on the context above:"""

In [20]:
sample_docs    = [doc for doc, _ in retrieve_relevant_chunks(test_query)]
sample_history = [
    {'role': 'user',      'content': 'What is AI?'},
    {'role': 'assistant', 'content': 'AI is the simulation of human intelligence by computers.'},
]

In [21]:
demo_prompt = build_prompt(test_query, sample_docs, sample_history)
print('Prompt preview (first 500 chars):')
print(demo_prompt[:500])
print(f'\nTotal prompt length: {len(demo_prompt)} chars')

Prompt preview (first 500 chars):
You are a precise, helpful AI Knowledge Assistant. Answer questions ONLY using the provided context. If the answer is not in the context, say: "I could not find relevant information in the uploaded documents." Do not fabricate information.

=== RETRIEVED CONTEXT ===
[Source 1: ai_overview.txt]
Deep Learning is a subset of ML using neural networks with many layers. It has revolutionized fields like computer vision, natural language processing, and speech recognition.

Natural Language Processing 

Total prompt length: 2621 chars


LLM Response Generation

In [22]:
# Option A: HuggingFace Flan-T5 — FREE, no API key needed
from transformers import pipeline as hf_pipeline, T5ForConditionalGeneration, T5Tokenizer

print('Loading google/flan-t5-base (downloads on first run)...')

Loading google/flan-t5-base (downloads on first run)...


In [23]:
# Load model and tokenizer directly for T5
model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')
tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4689.55it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [24]:
def generate_text_t5(prompt: str, max_length: int = 256) -> str:
    """Generate text using T5 model directly."""
    inputs = tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True,
        temperature=0.7
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print('Model ready')

Model ready


In [25]:
def generate_answer_hf(query: str, context_docs: list, chat_history: list) -> dict:
    """Generate answer using HuggingFace Flan-T5 (no API key needed)."""
    context = '\n\n'.join([d.page_content for d in context_docs])
    sources = list({d.metadata.get('source', 'Unknown') for d in context_docs})
    prompt  = f'Context: {context[:1500]}\n\nQuestion: {query}\n\nAnswer:'
    answer  = generate_text_t5(prompt, max_length=256)
    return {'answer': answer, 'sources': sources}


result = generate_answer_hf(test_query, sample_docs, sample_history)
print(f'\nQ: {test_query}')
print(f'A: {result["answer"]}')
print(f'Sources: {result["sources"]}')

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Q: What is RAG and how does it work?
A: Retrieval-Augmented Generation
Sources: ['ai_overview.txt']


Groq LLaMA3

In [26]:
GROQ_API_KEY = 'D:\Guvi\Mini Project 5\config.py'

def generate_answer_groq(query: str, context_docs: list, chat_history: list) -> dict:
    """Generate answer using Groq LLaMA3 (requires GROQ_API_KEY)."""
    if not GROQ_API_KEY:
        return {'answer': 'GROQ_API_KEY not set.', 'sources': []}
    try:
        from groq import Groq
        client  = Groq(api_key=GROQ_API_KEY)
        sources = list({d.metadata.get('source', 'Unknown') for d in context_docs})
        prompt  = build_prompt(query, context_docs, chat_history)
        resp    = client.chat.completions.create(
            model='llama-3.1-8b-instant',  # Updated to current supported model
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=1024,
            temperature=0.1,
        )
        return {'answer': resp.choices[0].message.content, 'sources': sources}
    except Exception as e:
        return {'answer': f'Groq error: {e}', 'sources': []}


if GROQ_API_KEY:
    r = generate_answer_groq(test_query, sample_docs, sample_history)
    print(f'A: {r["answer"]}')
else:
    print('ℹ️  Groq skipped — GROQ_API_KEY not set.')
    print('   Get free key: https://console.groq.com')

A: Groq error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


Conversational Chat Memory

In [27]:
class ChatMemory:
    """Manages conversation history with a configurable window."""

    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self.history   = []

    def add_message(self, role: str, content: str):
        self.history.append({'role': role, 'content': content})
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-(self.max_turns * 2):]

    def get_recent(self, n_turns=3):
        return self.history[-(n_turns * 2):]

    def clear(self):
        self.history = []


class RAGChatbot:
    """Full RAG chatbot: retrieval + prompt engineering + LLM."""

    def __init__(self, retriever, llm_fn, top_k=4):
        self.retriever = retriever
        self.llm_fn    = llm_fn
        self.top_k     = top_k
        self.memory    = ChatMemory()

    def chat(self, query: str) -> dict:
        docs    = self.retriever.invoke(query)
        history = self.memory.get_recent()
        result  = self.llm_fn(query, docs, history)
        self.memory.add_message('user', query)
        self.memory.add_message('assistant', result['answer'])
        result['chunks_retrieved'] = len(docs)
        return result

    def reset(self):
        self.memory.clear()
        print('🔄 Chat history cleared.')


Multi-turn demo

In [28]:
bot = RAGChatbot(retriever, generate_answer_hf, top_k=TOP_K)

demo_queries = [
    'What is machine learning?',
    'What are its main types?',
    'Tell me about deep learning.',
]


for q in demo_queries:
    print(f'\n USER: {q}')
    resp = bot.chat(q)
    print(f' BOT : {resp["answer"]}')
    print(f'    Sources: {resp["sources"]} | Chunks: {resp["chunks_retrieved"]}')


 USER: What is machine learning?
 BOT : ML is a subset of AI that allows systems to learn and improve from experience without being explicitly programmed
    Sources: ['ai_overview.txt'] | Chunks: 3

 USER: What are its main types?
 BOT : Supervised Learning:
    Sources: ['ai_overview.txt'] | Chunks: 3

 USER: Tell me about deep learning.
 BOT : Deep Learning is a subset of ML using neural networks with many layers
    Sources: ['ai_overview.txt'] | Chunks: 3


Evaluation Metrics

In [29]:
def evaluate_retrieval(queries: list, expected_keywords: list) -> dict:
    """Check if expected keywords appear in retrieved chunks."""
    total     = len(queries)
    hits      = 0
    latencies = []

    for query, keywords in zip(queries, expected_keywords):
        start   = time.time()
        results = retrieve_relevant_chunks(query, k=TOP_K)
        latencies.append(time.time() - start)

        retrieved_text = ' '.join([doc.page_content.lower() for doc, _ in results])
        matched        = sum(kw.lower() in retrieved_text for kw in keywords)
        if matched >= len(keywords) // 2 + 1:
            hits += 1

    return {
        'retrieval_accuracy': hits / total,
        'avg_latency_ms':     (sum(latencies) / len(latencies)) * 1000,
        'total_queries':      total,
        'hits':               hits,
    }

In [30]:
test_queries = [
    'What is artificial intelligence?',
    'What is supervised learning?',
    'How does NLP work?',
    'What is retrieval-augmented generation?',
]
expected_keywords = [
    ['intelligence', 'simulation', 'computer'],
    ['supervised', 'labeled', 'data'],
    ['language', 'computers', 'text'],
    ['retrieval', 'generation', 'knowledge'],
]

metrics = evaluate_retrieval(test_queries, expected_keywords)

print(f'Retrieval Accuracy : {metrics["retrieval_accuracy"]:.1%}')
print(f'Avg Retrieval Time : {metrics["avg_latency_ms"]:.1f} ms')
print(f'Total Test Queries : {metrics["total_queries"]}')
print(f'Successful Hits    : {metrics["hits"]}/{metrics["total_queries"]}')

Retrieval Accuracy : 100.0%
Avg Retrieval Time : 11.4 ms
Total Test Queries : 4
Successful Hits    : 4/4


Utility Functions

In [31]:
def save_uploaded_file(uploaded_bytes: bytes, filename: str, dest_dir: Path = DATASET_DIR) -> Path:
    """Save uploaded file bytes to the dataset directory."""
    dest = dest_dir / filename
    with open(dest, 'wb') as f:
        f.write(uploaded_bytes)
    print(f'✅ Saved: {dest}')
    return dest

In [32]:
def list_dataset_files(directory: Path = DATASET_DIR) -> list:
    """List all files in the dataset directory with metadata."""
    return sorted([
        {
            'name':      f.name,
            'size_kb':   round(f.stat().st_size / 1024, 2),
            'extension': f.suffix.lower(),
            'modified':  datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M'),
        }
        for f in directory.iterdir()
    ], key=lambda x: x['name'])

In [33]:
def rebuild_index(directory: Path = DATASET_DIR):
    """Full pipeline: load → chunk → embed → FAISS index."""
    docs = load_documents(directory)
    if not docs:
        return None, 0
    chunks_ = chunk_documents(docs)
    vs      = FAISS.from_documents(chunks_, embedding_model)
    return vs.as_retriever(search_kwargs={'k': TOP_K}), len(chunks_)

In [34]:
print('Dataset Contents:')
for f in list_dataset_files():
    print(f'  • {f["name"]:30s} {f["size_kb"]:6.1f} KB  {f["modified"]}')

Dataset Contents:
  • ai_overview.txt                   1.9 KB  2026-03-29 18:55
